# Bitcoin API — Core Component
This notebook demonstrates and explains the key functionality of the `bitcoin_utils.py` API module for real-time Bitcoin data ingestion and analysis using Apache Ray.

## 🔧 Initialization

In [ ]:

!pip install ray pandas requests

import ray
import time
import pandas as pd

ray.shutdown()
ray.init(ignore_reinit_error=True, include_dashboard=False)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 MB 11.5 MB/s eta 0:00:00


2025-04-30 15:34:08,210	INFO worker.py:1888 -- Started a local Ray instance.


Python version:,3.11.12
Ray version:,2.45.0


## 📦 Load API

In [2]:
from bitcoin_utils import fetch_bitcoin_price, PriceProcessor


2025-04-30 15:34:25,600	INFO worker.py:1888 -- Started a local Ray instance.


## fetch_bitcoin_price
Fetches real-time Bitcoin price from CoinGecko.

In [3]:
timestamp, price = ray.get(fetch_bitcoin_price.remote())
print(f"Fetched price: ${price} at {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(timestamp))}")


Fetched price: $93877 at 2025-04-30 15:34:30


## PriceProcessor — Ray Actor
Maintains internal state and performs time series analysis.

In [4]:
processor = PriceProcessor.remote()
ray.get(processor.add_price.remote(timestamp, price))
print(ray.get(processor.get_data.remote()))


[(1746027270.0080018, 93877)]


### Convert to Readable Timestamps

In [5]:
readable = ray.get(processor.get_data_with_readable_time.remote())
for entry in readable:
    print(entry)


{'timestamp': '2025-04-30 15:34:30', 'price': 93877}


###  Add More Data for Testing

In [6]:
for _ in range(4):
    ts, pr = ray.get(fetch_bitcoin_price.remote())
    ray.get(processor.add_price.remote(ts, pr))
    time.sleep(1)


###  Moving Average

In [7]:
moving_avg = ray.get(processor.compute_moving_average.remote(window=3))
for row in moving_avg:
    print(row)


{'timestamp': 1746027270.0080018, 'price': 93877, 'moving_avg': nan}
{'timestamp': 1746027277.990408, 'price': 93877, 'moving_avg': nan}
{'timestamp': 1746027279.0397882, 'price': 93877, 'moving_avg': 93877.0}
{'timestamp': 1746027280.104877, 'price': 93877, 'moving_avg': 93877.0}
{'timestamp': 1746027281.1486285, 'price': 93877, 'moving_avg': 93877.0}


### Percentage Price Changes

In [8]:
pct_changes = ray.get(processor.compute_percentage_changes.remote())
for row in pct_changes:
    print(row)


{'timestamp': '2025-04-30 15:34:37', 'price': 93877, 'percent_change': 0.0}
{'timestamp': '2025-04-30 15:34:39', 'price': 93877, 'percent_change': 0.0}
{'timestamp': '2025-04-30 15:34:40', 'price': 93877, 'percent_change': 0.0}
{'timestamp': '2025-04-30 15:34:41', 'price': 93877, 'percent_change': 0.0}


### Volatility Calculation

In [9]:
volatility = ray.get(processor.compute_volatility.remote(window=3))
for row in volatility:
    print(row)


{'timestamp': '2025-04-30 15:34:40', 'return': 0.0, 'volatility': 0.0}
{'timestamp': '2025-04-30 15:34:41', 'return': 0.0, 'volatility': 0.0}


### Filter Prices Above Threshold

In [10]:
high_prices = ray.get(processor.filter_prices_above.remote(50000))
for row in high_prices:
    print(row)


{'timestamp': '2025-04-30 15:34:30', 'price': 93877}
{'timestamp': '2025-04-30 15:34:37', 'price': 93877}
{'timestamp': '2025-04-30 15:34:39', 'price': 93877}
{'timestamp': '2025-04-30 15:34:40', 'price': 93877}
{'timestamp': '2025-04-30 15:34:41', 'price': 93877}
